## 最低限のデータの確認

In [1]:
import os


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/japan-ai-cup/sample_submission.csv
/kaggle/input/japan-ai-cup/train_flag.csv
/kaggle/input/japan-ai-cup/data.csv


In [2]:
import pandas as pd


data = pd.read_csv("../input/japan-ai-cup/data.csv")
data['date'] = pd.to_datetime(data['date'])
train_flag = pd.read_csv("../input/japan-ai-cup/train_flag.csv")
sub = pd.read_csv("../input/japan-ai-cup/sample_submission.csv")

In [3]:
# 最大表示列数の指定（ここでは50列を指定）
pd.set_option('display.max_columns', 50)

In [4]:
data.shape

(2757288, 24)

In [5]:
data.head()

,jan_cd,item_name,item_spec,item_category_cd_1,item_category_cd_2,item_category_cd_3,item_category_name,average_unit_price,amount,total_price,user_id,date,store_deli,user_flag_ec,membership_start_ym,age_category,sex,user_stage,user_flag_1,user_flag_2,user_flag_3,user_flag_4,user_flag_5,user_flag_6
0,4904230041160,ブラックニッカディープブレンド,７００ｍｌ,25,12,1,国産洋酒,1375.0,4,5500,59ad47dd,1970-01-01 00:00:00.020240203,店宅両方,0,201610.0,80代～,女性,メンバー,0,1,0,1,0,0
1,4901777284364,ＳＵ 角瓶ジャンボ,１９２０ｍｌ,25,12,1,国産洋酒,4895.0,1,4895,5079bb50,1970-01-01 00:00:00.020240203,店宅両方,0,201702.0,50代,女性,ゴールド,0,0,0,0,0,0
2,280743000000,寿司セット（景福）,１パック,18,34,3,セット,1078.0,4,4312,1720d922,1970-01-01 00:00:00.020240203,店舗のみ,0,202402.0,60代,女性,メンバー,0,1,0,0,0,0
3,247987000000,恵方巻（海鮮）,石狩１本,21,10,3,巻寿司,862.0,5,4310,178$c824,1970-01-01 00:00:00.020240203,店舗のみ,0,202304.0,40代,女性,メンバー,0,0,0,0,0,0
4,4902222001192,コープ北海道トドックブレンド（エージ）,５ｋｇ,24,1,1,白米道産米,1825.0,2,3650,55$a1a$b,1970-01-01 00:00:00.020240203,店舗のみ,0,200410.0,50代,女性,メンバー,0,0,0,0,0,0


In [6]:
train_flag.shape

(29965, 2)

In [7]:
train_flag.head()

,user_id,churn
0,$1$c92,1
1,$1cd7f,1
2,$1d3a9,1
3,$1f87d,1
4,$1fc60,1


In [8]:
sub.shape

(10000, 2)

In [9]:
sub.head()

,user_id,pred
0,$1d062,0.5
1,$5$ab$4,0.5
2,$5$f5af,0.5
3,$623182,0.5
4,$65b$2,0.5


In [10]:
data["user_id"].value_counts()

user_id
13bd0916    3220
30dcdd5b    2526
2bdf28ad    2306
179444c9    2165
18a0cc3$    2116
            ... 
15c$c837       1
1737fd20       1
13cd3c51       1
173671b4       1
1721b3$7       1
Name: count, Length: 40496, dtype: int64

In [11]:
train_flag["churn"].value_counts()

churn
1    22022
0     7943
Name: count, dtype: int64

## 簡易的なモデルの作成

In [12]:
import numpy as np
import scipy
from scipy import stats
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# Data Preprocessing
# ==========================================
print("Feature Engineering...")

# データの前処理
df_data = data.copy()
# dateが既にdatetime型の場合はそのまま使用、そうでない場合は変換
if not pd.api.types.is_datetime64_any_dtype(df_data['date']):
    df_data['date'] = pd.to_datetime(df_data['date'].astype(str), format='%Y%m%d')
df_data.sort_values(['user_id', 'date'], inplace=True)
df_data['is_weekend'] = (df_data['date'].dt.dayofweek >= 5).astype(int)
df_data['diff_days'] = df_data.groupby('user_id')['date'].diff().dt.days
df_data['membership_start_ym'] = df_data['membership_start_ym'].astype(str)

max_date = df_data['date'].max() + pd.Timedelta(days=1)

# Domain Adaptation (Hard Filtering)
print("\n=== Pre-Processing: Domain Adaptation (Hard Filtering) ===")
original_train_len = len(train_flag)
raw_users = df_data.sort_values('date').groupby('user_id').tail(1)[
    ['user_id', 'age_category', 'sex', 'membership_start_ym']
]
raw_users['join_year'] = raw_users['membership_start_ym'].str[:4]

train_users_ids = train_flag['user_id'].unique()
test_users_ids = sub['user_id'].unique()
df_users_train = raw_users[raw_users['user_id'].isin(train_users_ids)].copy()
df_users_test = raw_users[raw_users['user_id'].isin(test_users_ids)].copy()

def get_drop_ids(train_df, test_df, cols):
    def make_key(df): return df[cols].fillna('NaN').astype(str).agg('_'.join, axis=1)
    train_keys = make_key(train_df)
    test_keys = set(make_key(test_df).unique())
    return train_df[~train_keys.isin(test_keys)]['user_id'].tolist()

drop_list = []
drop_list += get_drop_ids(df_users_train, df_users_test, ['join_year'])
drop_list += get_drop_ids(df_users_train, df_users_test, ['age_category', 'sex'])
drop_list += get_drop_ids(df_users_train, df_users_test, ['join_year', 'age_category'])
drop_ids_unique = set(drop_list)

if len(drop_ids_unique) > 0:
    print(f"Candidates to drop: {len(drop_ids_unique)} users")
    train_flag = train_flag[~train_flag['user_id'].isin(drop_ids_unique)].reset_index(drop=True)
    print(f"Train Size: {original_train_len} -> {len(train_flag)} (Dropped {original_train_len - len(train_flag)})")
else:
    print("No users dropped based on domain mismatch.")

# ==========================================
# Feature Engineering
# ==========================================

# 返品・ゼロ取引の特徴
feat_dfs = []
for condition, prefix in [(df_data['amount'] < 0, 'return'), (df_data['amount'] == 0, 'zero')]:
    agg = df_data[condition].groupby('user_id').agg({
        'amount': ['count', 'sum'] if prefix == 'return' else ['count'],
        'total_price': ['sum']
    })
    agg.columns = [f"{prefix}_{c[0]}_{c[1]}" if c[1] else f"{prefix}_{c[0]}" for c in agg.columns]
    feat_dfs.append(agg)

# 正の取引のみで処理
df_clean = df_data[df_data['amount'] > 0].copy()

# 2月の取引回数
feb_mask = (df_clean['date'].dt.month == 2)
feb_feat = df_clean[feb_mask].groupby('user_id').size().rename('count_feb_all')

# バスケット単位の集約（日付ごと）
df_basket = df_clean.groupby(['user_id', 'date']).agg({
    'total_price': 'sum',
    'amount': 'count',
    'is_weekend': 'max'
}).reset_index().sort_values(['user_id', 'date'])
df_basket['diff_days'] = df_basket.groupby('user_id')['date'].diff().dt.days
global_diff_mean = df_basket['diff_days'].mean()

# 基本統計特徴
agg_base = {
    'date': ['count'],
    'total_price': ['sum', 'mean', 'max', 'min', 'std'],
    'amount': ['sum', 'mean'],
    'diff_days': ['mean', 'std', 'max', 'min'],
    'is_weekend': ['mean', 'sum']
}
df_features = df_basket.groupby('user_id').agg(agg_base)
df_features.columns = ['_'.join(col).strip() for col in df_features.columns.values]

# 時間窓特徴（30, 90, 180日）
agg_window = {'total_price': ['sum', 'count'], 'amount': ['mean'], 'diff_days': ['mean'], 'is_weekend': ['mean']}
for days in [30, 90, 180]:
    cutoff = max_date - pd.Timedelta(days=days)
    tmp = df_basket[df_basket['date'] >= cutoff].groupby('user_id').agg(agg_window)
    tmp.columns = [f"{c[0]}_{c[1]}_{days}d" for c in tmp.columns]
    df_features = df_features.join(tmp, how='left').fillna(0)

# 時系列特徴
date_stats = df_basket.groupby('user_id')['date'].agg(['max', 'min'])
df_features['recency'] = (max_date - date_stats['max']).dt.days
df_features['duration'] = (max_date - date_stats['min']).dt.days
df_features['is_one_time_buyer'] = (df_features['date_count'] == 1).astype(int)

# 時系列比率特徴
diff_mean_filled = df_features['diff_days_mean'].fillna(global_diff_mean)
df_features['recency_factor'] = df_features['recency'] / (diff_mean_filled + 1)
df_features['diff_days_max_ratio'] = df_features['diff_days_max'] / (df_features['diff_days_mean'] + 0.1)
df_features['recency_to_max_ratio'] = df_features['recency'] / (df_features['diff_days_max'] + 1)
df_features['trend_price_30_90'] = df_features['total_price_sum_30d'] / (df_features['total_price_sum_90d'] + 1)
df_features['ratio_visit_30_all'] = df_features['total_price_count_30d'] / (df_features['date_count'] + 1)
df_features['trend_diff_days'] = (
    df_features['diff_days_mean_30d'].replace(0, np.nan).fillna(global_diff_mean) / (diff_mean_filled + 0.01)
)

# Item2Vec
print("Generating Item2Vec Features...")
train_sentences = df_clean.groupby('user_id')['item_category_name'].apply(list).tolist()
model_w2v = Word2Vec(sentences=train_sentences, vector_size=32, window=5, min_count=1, workers=4, seed=42)

def get_user_vector(item_list):
    vectors = [model_w2v.wv[item] for item in item_list if item in model_w2v.wv]
    if len(vectors) == 0: return np.zeros(32)
    return np.mean(vectors, axis=0)

user_vectors = {}
for user_id, items in df_clean.groupby('user_id')['item_category_name']:
    user_vectors[user_id] = get_user_vector(items)

w2v_cols = [f'w2v_{i}' for i in range(32)]
df_w2v = pd.DataFrame.from_dict(user_vectors, orient='index', columns=w2v_cols)
df_w2v.index.name = 'user_id'
df_features = pd.merge(df_features, df_w2v, on='user_id', how='left')
df_features[w2v_cols] = df_features[w2v_cols].fillna(0)

# 最後3回の訪問に基づくトレンド特徴
last_3_visits = df_basket.sort_values(['user_id', 'date']).groupby('user_id').tail(3)
trend_df = last_3_visits.groupby('user_id')['diff_days'].agg(list).reset_index()

def calculate_slope(diff_list):
    if len(diff_list) < 3: return 0
    d1, d2, d3 = diff_list
    return d3 - d1

def calculate_ratio(diff_list):
    if len(diff_list) < 3: return 1.0
    d1, d2, d3 = diff_list
    avg = (d1 + d2 + d3) / 3 + 0.1
    return d3 / avg

trend_df['diff_days_slope'] = trend_df['diff_days'].apply(calculate_slope)
trend_df['diff_days_trend_ratio'] = trend_df['diff_days'].apply(calculate_ratio)
df_features = pd.merge(df_features, trend_df[['user_id', 'diff_days_slope', 'diff_days_trend_ratio']], on='user_id', how='left')
df_features['diff_days_slope'] = df_features['diff_days_slope'].fillna(0)
df_features['diff_days_trend_ratio'] = df_features['diff_days_trend_ratio'].fillna(1.0)

# メンバーシップ開始からの月数
member_start_map = df_clean.groupby('user_id')['membership_start_ym'].first()
def calculate_months_since(ym_value):
    try:
        ym_str = str(int(float(ym_value)))
        y, m = int(ym_str[:4]), int(ym_str[4:])
        return (2025 - y) * 12 + (2 - m)
    except:
        return 999

df_features['months_since_joined'] = df_features.index.map(member_start_map).map(calculate_months_since)
df_features['is_new_member'] = (df_features['months_since_joined'] <= 3).astype(int)
df_features['is_weekday_onetime'] = (
    (df_features['is_one_time_buyer'] == 1) & (df_features['is_weekend_mean'] == 0)
).astype(int)

# 行動特徴
df_features['avg_unit_price'] = df_features['total_price_mean'] / (df_features['amount_mean'].replace(0, 1))
df_basket['day'] = df_basket['date'].dt.day
payday_agg = df_basket.groupby('user_id')['day'].apply(lambda x: (x >= 25).mean())
df_features['payday_visit_ratio'] = df_features.index.map(payday_agg).fillna(0)

# 年齢・性別のインタラクション
user_profile = df_clean.sort_values('date').groupby('user_id').tail(1)[['user_id', 'age_category', 'sex']]
user_profile['age_sex_interact'] = (
    user_profile['age_category'].fillna('Unknown').astype(str) + '_' +
    user_profile['sex'].fillna('Unknown').astype(str)
)
df_features = pd.merge(df_features, user_profile[['user_id', 'age_sex_interact']], on='user_id', how='left')
le = LabelEncoder()
df_features['age_sex_interact'] = le.fit_transform(df_features['age_sex_interact'].astype(str))

# クラスタリング
cluster_cols_base = ['recency', 'date_count', 'total_price_mean', 'diff_days_mean']
if all(c in df_features.columns for c in cluster_cols_base):
    X_cluster = df_features[cluster_cols_base].copy()
    X_cluster['total_price_mean'] = np.log1p(X_cluster['total_price_mean'])
    X_cluster = X_cluster.fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    df_features['cluster_id'] = kmeans.fit_predict(X_scaled)

# カテゴリエントロピー
cat_counts = pd.crosstab(df_clean['user_id'], df_clean['item_category_name'])
cat_probs = cat_counts.div(cat_counts.sum(axis=1), axis=0)
entropy_series = pd.Series(scipy.stats.entropy(cat_probs.T), index=cat_counts.index)
df_features['cat_entropy'] = df_features.index.map(entropy_series).fillna(0)

# 定番商品の再購入
loyal_cats = ['菓子パン', '食パン', 'プレーンヨーグルト', '納豆', '牛乳', '野菜', '精肉', '鮮魚']
staple_logs = df_clean[df_clean['item_category_name'].isin(loyal_cats)]
last_staple_date = staple_logs.groupby('user_id')['date'].max()
df_features['staple_recency'] = df_features.index.map(lambda x: (max_date - last_staple_date.get(x, pd.NaT)).days)
df_features['staple_recency'] = df_features['staple_recency'].fillna(365)

# ルーティン強度
def calculate_routine_strength(x):
    if len(x) < 3: return 0.5
    return x.value_counts(normalize=True).max()
df_basket['dow'] = df_basket['date'].dt.dayofweek
routine_map = df_basket.groupby('user_id')['dow'].apply(calculate_routine_strength)
df_features['routine_strength'] = df_features.index.map(routine_map).fillna(0.5)

# 全ての特徴をマージ
df_features = df_features.join(feat_dfs, how='left')
df_features = df_features.join(feb_feat, how='left')
df_features['count_feb_all'] = df_features['count_feb_all'].fillna(0)
df_features['is_returner'] = (df_features.get('return_amount_count', 0) > 0).astype(int)
df_features['is_zero_user'] = (df_features.get('zero_amount_count', 0) > 0).astype(int)

# プロファイル特徴
df_profile = df_clean.sort_values('date').groupby('user_id').tail(1)[
    ['user_id', 'age_category', 'sex', 'membership_start_ym', 'user_stage']
].set_index('user_id')
cols_to_use = df_profile.columns.difference(df_features.columns)
df_features = df_features.join(df_profile[cols_to_use], how='left')

# カテゴリ特徴のエンコード
for col in ['age_category', 'sex', 'membership_start_ym', 'user_stage']:
    if col in df_features.columns:
        filled_col = df_features[col].fillna('NaN').astype(str)
        df_features[col] = LabelEncoder().fit_transform(filled_col)

# ==========================================
# 追加特徴量: store_deli関連（最優先）
# ==========================================
if 'store_deli' in df_clean.columns:
    store_deli_stats = df_clean.groupby('user_id')['store_deli'].agg(['count', lambda x: (x == '店舗のみ').sum(), lambda x: (x == '宅配のみ').sum(), lambda x: (x == '店宅両方').sum()])
    store_deli_stats.columns = ['store_deli_count', 'store_only_count', 'deli_only_count', 'both_count']
    df_features['store_ratio'] = store_deli_stats['store_only_count'] / (store_deli_stats['store_deli_count'] + 1)
    df_features['deli_ratio'] = store_deli_stats['deli_only_count'] / (store_deli_stats['store_deli_count'] + 1)
    df_features['is_store_only'] = (store_deli_stats['store_only_count'] == store_deli_stats['store_deli_count']).astype(int)
    df_features['is_deli_only'] = (store_deli_stats['deli_only_count'] == store_deli_stats['store_deli_count']).astype(int)
    store_deli_sequence = df_clean.sort_values(['user_id', 'date']).groupby('user_id')['store_deli'].apply(list)
    def count_switches(seq):
        if len(seq) < 2: return 0
        switches = 0
        for i in range(1, len(seq)):
            if seq[i] != seq[i-1]:
                switches += 1
        return switches
    df_features['store_deli_switch_count'] = df_features.index.map(lambda x: count_switches(store_deli_sequence.get(x, []))).fillna(0)
    last_channel = df_clean.sort_values(['user_id', 'date']).groupby('user_id')['store_deli'].last()
    df_features['last_purchase_channel'] = df_features.index.map(lambda x: 0 if last_channel.get(x, '') == '店舗のみ' else (1 if last_channel.get(x, '') == '宅配のみ' else 0.5)).fillna(0)

# ==========================================
# 追加特徴量: user_flag関連（第2優先）
# ==========================================
if 'user_flag_1' in df_clean.columns:
    flag_cols = [f'user_flag_{i}' for i in range(1, 7) if f'user_flag_{i}' in df_clean.columns]
    if len(flag_cols) > 0:
        user_flags = df_clean.sort_values(['user_id', 'date']).groupby('user_id')[flag_cols].first()
        df_features['flag_sum'] = user_flags.sum(axis=1)
        df_features['flag_active_count'] = (user_flags == 1).sum(axis=1)
        flag_combination = user_flags.astype(str).agg('_'.join, axis=1)
        le_flag = LabelEncoder()
        df_features['flag_combination'] = le_flag.fit_transform(flag_combination.fillna('0_0_0_0_0_0'))
        df_features = df_features.join(user_flags, how='left')
if 'user_flag_ec' in df_clean.columns:
    user_flag_ec = df_clean.sort_values(['user_id', 'date']).groupby('user_id')['user_flag_ec'].first()
    df_features['user_flag_ec'] = df_features.index.map(user_flag_ec).fillna(0)

# ==========================================
# 追加特徴量: カテゴリコード多様性（第3優先）
# ==========================================
if 'item_category_cd_1' in df_clean.columns:
    df_features['category_cd1_diversity'] = df_clean.groupby('user_id')['item_category_cd_1'].nunique()
    most_freq_cd1 = df_clean.groupby('user_id')['item_category_cd_1'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else -1)
    le_cd1 = LabelEncoder()
    df_features['most_frequent_cd1'] = le_cd1.fit_transform(most_freq_cd1.fillna(-1).astype(str))
if 'item_category_cd_2' in df_clean.columns:
    df_features['category_cd2_diversity'] = df_clean.groupby('user_id')['item_category_cd_2'].nunique()
if 'item_category_cd_3' in df_clean.columns:
    df_features['category_cd3_diversity'] = df_clean.groupby('user_id')['item_category_cd_3'].nunique()
    cd3_counts = df_clean.groupby(['user_id', 'item_category_cd_3']).size().reset_index(name='count')
    cd3_totals = cd3_counts.groupby('user_id')['count'].sum()
    cd3_probs = cd3_counts.set_index('user_id')['count'] / cd3_totals
    cd3_entropy_by_user = cd3_probs.groupby('user_id').apply(lambda x: scipy.stats.entropy(x.values))
    df_features['category_cd_entropy'] = df_features.index.map(cd3_entropy_by_user).fillna(0)

# ==========================================
# 追加特徴量: 商品多様性（第4優先）
# ==========================================
if 'jan_cd' in df_clean.columns:
    df_features['unique_products_count'] = df_clean.groupby('user_id')['jan_cd'].nunique()
    product_counts = df_clean.groupby(['user_id', 'jan_cd']).size().reset_index(name='count')
    product_totals = product_counts.groupby('user_id')['count'].sum()
    product_probs = product_counts.set_index('user_id')['count'] / product_totals
    product_entropy_by_user = product_probs.groupby('user_id').apply(lambda x: scipy.stats.entropy(x.values))
    df_features['product_diversity'] = df_features.index.map(product_entropy_by_user).fillna(0)
    product_purchase_count = df_clean.groupby(['user_id', 'jan_cd']).size()
    repeat_products = product_purchase_count[product_purchase_count > 1].reset_index()['user_id'].value_counts()
    total_products = df_clean.groupby('user_id')['jan_cd'].nunique()
    df_features['repeat_purchase_ratio'] = df_features.index.map(lambda x: repeat_products.get(x, 0) / (total_products.get(x, 1) + 1e-6)).fillna(0)
if 'item_spec' in df_clean.columns:
    df_features['item_spec_diversity'] = df_clean.groupby('user_id')['item_spec'].nunique()

# ==========================================
# 追加特徴量: 時系列一貫性（第5優先）
# ==========================================
if 'diff_days_std' in df_features.columns:
    df_features['visit_consistency'] = 1 / (df_features['diff_days_std'] + 1)
if 'total_price_std' in df_features.columns and 'total_price_mean' in df_features.columns:
    df_features['price_volatility'] = df_features['total_price_std'] / (df_features['total_price_mean'] + 1e-6)
if 'date_count' in df_features.columns:
    mid_date = df_basket['date'].quantile(0.5)
    first_half_basket = df_basket[df_basket['date'] <= mid_date].groupby('user_id')['amount'].mean()
    second_half_basket = df_basket[df_basket['date'] > mid_date].groupby('user_id')['amount'].mean()
    basket_trend = second_half_basket / (first_half_basket + 1)
    df_features['basket_size_trend'] = df_features.index.map(basket_trend).fillna(1.0)

# 欠損値・異常値の処理
num_cols = df_features.select_dtypes(include=[np.number]).columns
df_features[num_cols] = df_features[num_cols].fillna(0).replace([np.inf, -np.inf], 0)

# 外れ値のクリッピング
for col in num_cols:
    if col != 'user_id':
        q1 = df_features[col].quantile(0.01)
        q99 = df_features[col].quantile(0.99)
        df_features[col] = df_features[col].clip(lower=q1, upper=q99)

# 最終的な特徴データフレーム
features = df_features.reset_index()
if 'index' in features.columns:
    features.drop(columns=['index'], inplace=True)

print(f"Feature Engineering Complete. Total features: {len(features.columns) - 1}")

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


Feature Engineering...

=== Pre-Processing: Domain Adaptation (Hard Filtering) ===
Candidates to drop: 49 users
Train Size: 29965 -> 29916 (Dropped 49)
Generating Item2Vec Features...
Feature Engineering Complete. Total features: 94


In [13]:
features.head()

,user_id,date_count,total_price_sum,total_price_mean,total_price_max,total_price_min,total_price_std,amount_sum,amount_mean,diff_days_mean,diff_days_std,diff_days_max,diff_days_min,is_weekend_mean,is_weekend_sum,total_price_sum_30d,total_price_count_30d,amount_mean_30d,diff_days_mean_30d,is_weekend_mean_30d,total_price_sum_90d,total_price_count_90d,amount_mean_90d,diff_days_mean_90d,is_weekend_mean_90d,...,w2v_31,diff_days_slope,diff_days_trend_ratio,months_since_joined,is_new_member,is_weekday_onetime,avg_unit_price,payday_visit_ratio,age_sex_interact,cluster_id,cat_entropy,staple_recency,routine_strength,return_amount_count,return_amount_sum,return_total_price_sum,zero_amount_count,zero_total_price_sum,count_feb_all,is_returner,is_zero_user,age_category,membership_start_ym,sex,user_stage
0,$1$c92,1.0,1226.0,1226.0,1226.0,1226.0,0.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,1226.0,1.0,7.0,0.0,0.0,1226.0,1.0,7.0,0.0,0.0,...,-1.060943,0.0,1.0,999.0,0.0,1.0,175.142857,0.0,10.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
1,$1cd7f,1.0,2735.0,2735.0,2735.0,2735.0,0.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,2735.0,1.0,6.0,0.0,0.0,2735.0,1.0,6.0,0.0,0.0,...,-1.011856,0.0,1.0,999.0,0.0,1.0,455.833333,0.0,17.0,1.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2,$1d062,1.0,276.0,276.0,276.0,276.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,276.0,1.0,2.0,0.0,0.0,276.0,1.0,2.0,0.0,0.0,...,0.623446,0.0,1.0,999.0,0.0,1.0,138.000000,0.0,3.0,4.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
3,$1d3a9,1.0,1789.0,1789.0,1789.0,1789.0,0.0,11.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,1789.0,1.0,11.0,0.0,0.0,1789.0,1.0,11.0,0.0,0.0,...,-0.368916,0.0,1.0,999.0,0.0,1.0,162.636364,0.0,4.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
4,$1f87d,1.0,1098.0,1098.0,1098.0,1098.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1098.0,1.0,1.0,0.0,0.0,1098.0,1.0,1.0,0.0,0.0,...,-0.890904,0.0,1.0,999.0,0.0,1.0,1076.000000,0.0,3.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0


In [14]:
X_train = pd.merge(train_flag, features, on="user_id", how="left")
X_train = X_train.drop(["user_id", "churn"], axis=1)
y_train = train_flag["churn"].values

In [15]:
X_train.head()

,date_count,total_price_sum,total_price_mean,total_price_max,total_price_min,total_price_std,amount_sum,amount_mean,diff_days_mean,diff_days_std,diff_days_max,diff_days_min,is_weekend_mean,is_weekend_sum,total_price_sum_30d,total_price_count_30d,amount_mean_30d,diff_days_mean_30d,is_weekend_mean_30d,total_price_sum_90d,total_price_count_90d,amount_mean_90d,diff_days_mean_90d,is_weekend_mean_90d,total_price_sum_180d,...,w2v_31,diff_days_slope,diff_days_trend_ratio,months_since_joined,is_new_member,is_weekday_onetime,avg_unit_price,payday_visit_ratio,age_sex_interact,cluster_id,cat_entropy,staple_recency,routine_strength,return_amount_count,return_amount_sum,return_total_price_sum,zero_amount_count,zero_total_price_sum,count_feb_all,is_returner,is_zero_user,age_category,membership_start_ym,sex,user_stage
0,1.0,1226.0,1226.0,1226.0,1226.0,0.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,1226.0,1.0,7.0,0.0,0.0,1226.0,1.0,7.0,0.0,0.0,1226.0,...,-1.060943,0.0,1.0,999.0,0.0,1.0,175.142857,0.0,10.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
1,1.0,2735.0,2735.0,2735.0,2735.0,0.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,2735.0,1.0,6.0,0.0,0.0,2735.0,1.0,6.0,0.0,0.0,2735.0,...,-1.011856,0.0,1.0,999.0,0.0,1.0,455.833333,0.0,17.0,1.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2,1.0,1789.0,1789.0,1789.0,1789.0,0.0,11.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,1789.0,1.0,11.0,0.0,0.0,1789.0,1.0,11.0,0.0,0.0,1789.0,...,-0.368916,0.0,1.0,999.0,0.0,1.0,162.636364,0.0,4.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
3,1.0,1098.0,1098.0,1098.0,1098.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1098.0,1.0,1.0,0.0,0.0,1098.0,1.0,1.0,0.0,0.0,1098.0,...,-0.890904,0.0,1.0,999.0,0.0,1.0,1076.000000,0.0,3.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
4,1.0,214.0,214.0,214.0,214.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,214.0,1.0,1.0,0.0,0.0,214.0,1.0,1.0,0.0,0.0,214.0,...,-2.773587,0.0,1.0,999.0,0.0,1.0,214.000000,0.0,3.0,4.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0


In [16]:
X_test = pd.merge(sub, features, on="user_id", how="left")
X_test = X_test.drop(["user_id", "pred"], axis=1)

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("="*50)
print("LightGBM Training")
print("="*50)

y_preds_lgb = []
models_lgb = []
oof_train_lgb = np.zeros((len(X_train),))
fold_scores_lgb = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

categorical_features = []

params_lgb = {
    "objective": "binary",
    "max_bin": 300,
    "learning_rate": 0.1,
    "num_leaves": 40,
    "metric": "auc"
}

for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    X_tr = X_train.loc[train_index, :]
    X_val = X_train.loc[valid_index, :]
    y_tr = y_train[train_index]
    y_val = y_train[valid_index]

    lgb_train = lgb.Dataset(X_tr, y_tr,
                            categorical_feature=categorical_features)
    lgb_eval = lgb.Dataset(X_val, y_val,
                           reference=lgb_train,
                           categorical_feature=categorical_features)

    model = lgb.train(params, lgb_train,
                      valid_sets=[lgb_train, lgb_eval],
                      num_boost_round=10000,
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(period=100)])

    oof_train_lgb[valid_index] = model.predict(X_val, num_iteration=model.best_iteration)
    
    # 各foldのスコアを計算
    fold_score = roc_auc_score(y_val, oof_train_lgb[valid_index])
    fold_scores_lgb.append(fold_score)
    print(f"LightGBM Fold {fold_id + 1} AUC: {fold_score:.6f}")
    
    models_lgb.append(model)
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)
    y_preds_lgb.append(y_pred)

# 全foldの統計情報を表示
print("\n" + "="*50)
print("LightGBM Cross-Validation Results:")

print("="*50)
for i, score in enumerate(fold_scores_lgb, 1):
    print(f"Fold {i} AUC: {score:.6f}")
print(f"\nMean AUC: {np.mean(fold_scores_lgb):.6f}")
print(f"Std AUC: {np.std(fold_scores_lgb):.6f}")
print("="*50)


In [ ]:
import xgboost as xgb

y_preds_xgb = []
models_xgb = []
oof_train_xgb = np.zeros((len(X_train),))
fold_scores_xgb = []

params_xgb = {
    "objective": "binary:logistic",
    "learning_rate": 0.1,
    "max_depth": 6,
    "eval_metric": "auc"
}

print("\n" + "="*50)
print("XGBoost Training")
print("="*50)

for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    X_tr = X_train.loc[train_index, :]
    X_val = X_train.loc[valid_index, :]
    y_tr = y_train[train_index]
    y_val = y_train[valid_index]

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params_xgb,
        dtrain,
        num_boost_round=10000,
        evals=[(dtrain, "train"), (dval, "eval")],
        early_stopping_rounds=100,
        verbose_eval=100
    )

    oof_train_xgb[valid_index] = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    
    fold_score = roc_auc_score(y_val, oof_train_xgb[valid_index])
    fold_scores_xgb.append(fold_score)
    print(f"XGBoost Fold {fold_id + 1} AUC: {fold_score:.6f}")
    
    models_xgb.append(model)
    dtest = xgb.DMatrix(X_test)
    y_pred = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
    y_preds_xgb.append(y_pred)

print("\n" + "="*50)
print("XGBoost Cross-Validation Results:")
print("="*50)
for i, score in enumerate(fold_scores_xgb, 1):
    print(f"Fold {i} AUC: {score:.6f}")
print(f"\nMean AUC: {np.mean(fold_scores_xgb):.6f}")
print(f"Std AUC: {np.std(fold_scores_xgb):.6f}")
print("="*50)


In [17]:
from sklearn.metrics import roc_auc_score

print("="*50)
print("Individual Model Scores:")
print("="*50)
lgb_score = roc_auc_score(y_train, oof_train_lgb)
xgb_score = roc_auc_score(y_train, oof_train_xgb)
print(f"LightGBM OOF AUC: {lgb_score:.6f}")
print(f"XGBoost OOF AUC: {xgb_score:.6f}")

print("\n" + "="*50)
print("Ensemble Score:")
print("="*50)
oof_ensemble = (oof_train_lgb + oof_train_xgb) / 2
ensemble_score = roc_auc_score(y_train, oof_ensemble)
print(f"Ensemble OOF AUC: {ensemble_score:.6f}")
print("="*50)


,date_count,total_price_sum,total_price_mean,total_price_max,total_price_min,total_price_std,amount_sum,amount_mean,diff_days_mean,diff_days_std,diff_days_max,diff_days_min,is_weekend_mean,is_weekend_sum,total_price_sum_30d,total_price_count_30d,amount_mean_30d,diff_days_mean_30d,is_weekend_mean_30d,total_price_sum_90d,total_price_count_90d,amount_mean_90d,diff_days_mean_90d,is_weekend_mean_90d,total_price_sum_180d,...,w2v_31,diff_days_slope,diff_days_trend_ratio,months_since_joined,is_new_member,is_weekday_onetime,avg_unit_price,payday_visit_ratio,age_sex_interact,cluster_id,cat_entropy,staple_recency,routine_strength,return_amount_count,return_amount_sum,return_total_price_sum,zero_amount_count,zero_total_price_sum,count_feb_all,is_returner,is_zero_user,age_category,membership_start_ym,sex,user_stage
0,1.0,276.0,276.000000,276.0,276.0,0.000000,2.0,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,276.0,1.0,2.000000,0.0,0.0,276.0,1.0,2.000000,0.0,0.0,276.0,...,0.623446,0.0,1.0,999.0,0.0,1.0,138.000000,0.0,3.0,4.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
1,1.0,1764.0,1764.000000,1764.0,1764.0,0.000000,7.0,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,1764.0,1.0,7.000000,0.0,0.0,1764.0,1.0,7.000000,0.0,0.0,1764.0,...,-0.908875,0.0,1.0,999.0,0.0,1.0,252.000000,0.0,19.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2,2.0,1923.0,961.500000,1827.0,96.0,1224.001838,11.0,5.500000,0.0,0.0,0.0,0.0,0.0,0.0,1923.0,2.0,5.500000,0.0,0.0,1923.0,2.0,5.500000,0.0,0.0,1923.0,...,-0.365782,0.0,1.0,999.0,0.0,0.0,174.818182,0.0,15.0,0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
3,2.0,4831.0,2415.500000,3944.0,887.0,2161.625430,12.0,6.000000,0.0,0.0,0.0,0.0,0.0,0.0,4831.0,2.0,6.000000,0.0,0.0,4831.0,2.0,6.000000,0.0,0.0,4831.0,...,0.232034,0.0,1.0,999.0,0.0,0.0,402.583333,0.0,13.0,1.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0
4,3.0,12166.0,4055.333333,5121.0,3144.0,997.495029,38.0,12.666667,0.0,0.0,0.0,0.0,0.0,0.0,12166.0,3.0,12.666667,0.0,0.0,12166.0,3.0,12.666667,0.0,0.0,12166.0,...,-0.440831,0.0,1.0,999.0,0.0,0.0,320.157895,0.0,10.0,1.0,0.0,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0


In [18]:
# アンサンブル予測（LightGBMとXGBoostの平均）
y_sub_lgb = sum(y_preds_lgb) / len(y_preds_lgb)
y_sub_xgb = sum(y_preds_xgb) / len(y_preds_xgb)
y_sub = (y_sub_lgb + y_sub_xgb) / 2

sub["pred"] = y_sub
sub.to_csv("submission_ensemble_lgb_xgb.csv", index=False)

sub.head()


[LightGBM] [Info] Number of positive: 17588, number of negative: 6344
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007828 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14741
[LightGBM] [Info] Number of data points in the train set: 23932, number of used features: 58
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.734916 -> initscore=1.019707
[LightGBM] [Info] Start training from score 1.019707
Training until validation scores don't improve for 100 rounds
[100]	training's auc: 0.963368	valid_1's auc: 0.889556
Early stopping, best iteration is:
[38]	training's auc: 0.926912	valid_1's auc: 0.891759
Fold 1 AUC: 0.891759
[LightGBM] [Info] Number of positive: 17588, number of negative: 6345
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006717 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14741
[LightGBM] [In

## 検証スコアの算出

In [19]:
from sklearn.metrics import roc_auc_score


roc_auc_score(y_train, oof_train)

np.float64(0.8746596203719441)

## 提出ファイルの作成

In [20]:
y_sub = sum(y_preds) / len(y_preds)
sub["pred"] = y_sub
sub.to_csv("submission_lightgbm_skfold.csv", index=False)

sub.head()

,user_id,pred
0,$1d062,0.908844
1,$5$ab$4,0.908675
2,$5$f5af,0.845822
3,$623182,0.826419
4,$65b$2,0.792298
